# SentinelAI — Phase 2 (Colab)

Trains the TF-IDF baseline and DistilBERT classifier end-to-end on Google Colab.

**Before you run:** `Runtime → Change runtime type → T4 GPU`.

Total runtime on free T4: ~10 minutes.

## What this notebook does
1. Installs dependencies
2. Uploads the SentinelAI repo (or clones it from GitHub)
3. Builds Phase 1 dataset (seed + public sources)
4. Trains TF-IDF baseline
5. Fine-tunes DistilBERT
6. Compares the two models
7. Downloads everything as a zip

## 0. Sanity check the GPU

In [ ]:
!nvidia-smi

## 1. Install dependencies

In [ ]:
!pip -q install transformers==4.46.0 datasets==2.21.0 scikit-learn==1.5.2 joblib==1.4.2 accelerate==1.0.1

## 2. Get the project

**Option A — clone from your GitHub repo (preferred once you push):**

```python
!git clone https://github.com/YOUR_USERNAME/sentinelai.git
%cd sentinelai
```

**Option B — upload the Phase 2 zip directly to this Colab session:**

In [ ]:
from google.colab import files
uploaded = files.upload()         # pick sentinelai-phase2.zip
!unzip -q -o sentinelai-phase2.zip -d /content/
%cd /content/sentinelai-phase2

## 3. Build dataset_v1

This pulls JailbreakBench, AdvBench, HH-RLHF and the verazuo forbidden-question set, then merges + deduplicates + balances + splits.

In [ ]:
!python scripts/build_seed_dataset.py
!python scripts/fetch_public_sources.py
!python scripts/build_dataset_v1.py

In [ ]:
import pandas as pd, json
df = pd.read_csv('data/dataset_v1.csv')
print(f'rows: {len(df):,}')
print(df['label'].value_counts())
print(df['attack_type'].value_counts())

## 4. Train the TF-IDF baseline

Establishes the 'traditional NLP' floor for RQ2. Should take <30 seconds.

In [ ]:
!cd scripts && python train_baseline.py

## 5. Fine-tune DistilBERT

Should take ~5–8 minutes on a T4.

In [ ]:
!cd scripts && python train_distilbert.py --epochs 4 --batch-size 16

### Optional: try RoBERTa-base for a stronger comparison

Roughly 2× slower than DistilBERT but usually ~1–2 F1 points higher.
Useful as an ablation in the thesis ("DistilBERT trades a small amount of
quality for runtime efficiency").

In [ ]:
# Uncomment to run
# !cd scripts && python train_distilbert.py \
#     --model roberta-base \
#     --run-name roberta_v1 \
#     --epochs 4 --batch-size 16 --learning-rate 1e-5

## 6. Compare models side by side

In [ ]:
import sys, json
sys.path.insert(0, 'scripts')
from evaluation import EvaluationReport, HeadlineMetrics, GroupedRecall, compare_reports
from pathlib import Path

def load_report(p):
    d = json.loads(Path(p).read_text())
    return EvaluationReport(
        model_name=d['model_name'],
        headline=HeadlineMetrics(**d['headline']),
        confusion_matrix=d['confusion_matrix'],
        per_attack_type=GroupedRecall(by_group=d['per_attack_type']),
        per_source=GroupedRecall(by_group=d['per_source']),
        per_topic_top10=GroupedRecall(by_group=d['per_topic_top10']),
        calibration=d['calibration'],
        top_misclassified=d['top_misclassified'],
    )

reports = [
    load_report('reports/baseline_tfidf_lr_report.json'),
    load_report('reports/distilbert_v1_report.json'),
]
comparison = compare_reports(reports)
print(comparison)
Path('reports/comparison.md').write_text(comparison)

## 7. Inspect per-attack-type recall (RQ3)

This is the table that drives your RQ3 chapter.  Disguise-class recall tells
you which kinds of attacks the transformer detects better than the baseline
and which still sneak through.

In [ ]:
import json
import pandas as pd

rows = []
for name, p in [('baseline', 'reports/baseline_tfidf_lr_report.json'),
                ('distilbert', 'reports/distilbert_v1_report.json')]:
    d = json.loads(open(p).read())
    for atk, m in d['per_attack_type'].items():
        rows.append({
            'model': name, 'attack_type': atk, 'support': m['support'],
            'recall_pos': m['recall_on_positives'],
            'specificity_neg': m['specificity_on_negatives'],
        })
df = pd.DataFrame(rows)
print(df.pivot_table(index='attack_type', columns='model',
                     values='recall_pos').round(3))

## 8. Download artefacts

Bundle everything you need (trained model, reports, splits) and pull it down.

In [ ]:
!zip -qr /content/sentinelai-phase2-trained.zip \
    models/ reports/ data/dataset_v1*.csv data/dataset_v1_summary.json
from google.colab import files
files.download('/content/sentinelai-phase2-trained.zip')